# Fuel versus fare: The effect of relative travel costs on modal split

Authors: Daan Devenijns, Maarten Hogendoorn, Hidde Okhuijsen, Gijs Six, Nova Swanenburg 

# 1. Introduction

The transition towards sustainable mobility requires a clear understanding of what drives travellers to choose public transport over private vehicles. While factors such as travel time, reliability, and comfort play an important role, the cost of travel remains one of the primary drivers behind commuting decisions. Over recent years, high volatility in fuel prices, combined with rising railway operational costs and fare adjustments, has continuously shifted the relative cost competitiveness between driving and taking the train.

This study investigates the impact of the ratio between fuel prices and train ticket prices on travellers' mode choice (car vs train). Rather than analysing the costs of a single mode, looking at the relative price ratio offers direct insight into the tipping point where passengers choose to change their mode of travel, without having to account for inflation.

# 2. Research Objective

The objective of this project is to quantify the impact of changes in fuel prices as well as train ticket prices on passenger mode choice, and predict future behaviour by estimating price elasticities.

#### Research question:
_What is the influence of the price ratio between fuel and train tickets on the choice between car and train?_

#### Sub questions:
* How has the ratio between fuel prices and train ticket prices evolved over the last 25 years?
* To what extent does a change in the fuel-to-fare price ratio affect the modal split between car and train?
* What is the price elasticity of modal choice with respect to the fuel-fare ratio?

#### Geografical and temporal scale:
To evaluate the relationship between transport pricing and public transport usage, this study examines the Netherlands from 1999 to 2025. This timeframe is determined by the availability of consistent, long-term longitudinal data from Statistics Netherlands (CBS) regarding mode usage and the Consumer Price Index (CPI) for both car and train transport.

# 3. Data Pipeline

To compare the relative price of rail travel to fuel costs, data from CBS will be used. Specifically, table 83131NED (*Consumentenprijzen; prijsindex 2015=100, 1996–2025*) is selected. This table provides the Consumer Price Index (CPI) across various expenditure categories, including passenger transport by train and different fuel types. 

To determine the modal split, CBS table 84755NED (*Mobiliteitstrend; per rit, vervoerwijzen, reismotief, leeftijd en geslacht*) will be used. This dataset covers the period from 1999 to 2024 and contains the average number of trips made per person per transport mode, alongside the average trip distance.

To control for changes in purchasing power, real disposable household income per capita is constructed from three additional CBS tables: 85881NED (*Kerngegevens sectoren; nationale rekeningen*) for gross disposable household income, 85524NED (*Bevolking, huishoudens en bevolkingsontwikkeling; vanaf 1899*) for the total population, and the overall CPI (category 000000, *Alle bestedingen*) from table 83131NED to correct for inflation.

This section outlines the definition and function of the Consumer Price Index (CPI), followed by an explanation of the data cleaning and processing pipeline implemented in Python.

## 3.1 Consumer Price Index (CPI)
The Consumer Price Index (CPI) measures the average price change of a representative basket of goods and services consumed by households. It is used as a tool to track yearly inflation and consumer purchasing power. For example, if the CPI rises from 100 in one year to 110 the next year, the representative basket has become 10 per cent more expensive.

The CBS data divides the CPI into main categories such as transport, education, or restaurants, and into subcategories for specific goods and services such as food, clothing, or fuel. The values of these subcategories can be used to assess the relative costs of products or services in that category. This study uses the CPI of passenger transport by train (subcategory 073110), diesel (072210), petrol (072220), and LPG (072230) to evaluate how the relative costs of train transport compare to the costs of different fuel types over time. Because both price series are normalised to the same base year (where the CPI in 2015 is set to 100), comparing the rail CPI directly to the fuel CPI filters out general inflation.

## 3.2 Data Processing Pipeline
This section explains how the raw data from CBS was processed into usable data using Python and the `pandas` package. The code can be found in the project repository in the notebook `code/data_pipeline.ipynb`, and the resulting clean datasets are saved in the folder `data/processed_data/`.

### 3.2.1 Processing CPI Data
The function `process_cpi_data` imports and standardises the raw CPI tables, combining the rail and fuel subcategories into a single DataFrame:
- **File import and cleaning**: For each input CSV file, the initial headers (the first four rows) and the last explanatory row are removed because they only contain metadata.
- **Date standardisation**: Dutch month names are mapped to two-digit numerical representations and converted to a standard `datetime` format for easier plotting later.
- **Type checking**: Decimal commas are replaced with points, and values are converted to floats (floating point numbers) to validate the data and enable numerical computations.
- **Concatenation and export**: The individual CPI series are combined into a single DataFrame, and exported to the processed data directory using the `export_dataframe` function.

The raw data table and resulting processed data table are shown below.

[Insert: Before and after screenshots of the CPI dataset]

### 3.2.2 Processing Modal Split and Travel Behaviour Data
The `process_mode_data` function imports and cleans the modal split dataset:
- **File import and cleaning**: The initial metadata headers (the first five rows) and the last explanatory row are removed because they only contain metadata. Asterisks indicating provisional annual figures are removed.
- **Translation**: All Dutch labels for travel purposes and transport modes are translated into English and the separate with the units (km, trips, min) are added directly into row with transport modes.
- **Date standardisation**: All years in the first column are converted to a standard `datetime` format for easier plotting later.
- **Type checking**: Decimal commas are replaced with points, and values are converted to floats (floating point numbers) to validate the data and enable numerical computations.
- **Calculation additional values**: 

- Create a combined 'Car (total)' category:
  - Calculate the total trips per person per day by summing driver and passenger trips.
  - Calculate the average car travel time per trip (weighted by the number of trips).
  - Calculate the average car travel distance per trip (weighted by the number of trips).
- Calculate the average daily distance travelled and average daily travel time for each mode by multiplying trips per person per day by the respective average trip metric.
- **Final structuring and export**: Columns are sorted alphabetically by mode, and the processed dataset is saved to `data/processed_data/` via `export_dataframe`.

The raw data table and resulting processed data table are shown below.

[Insert: Before and after screenshots of the mobility survey dataset]

### 3.2.3 Processing Income Data
The function `process_income_data` combines three CBS tables into a single annual series of real disposable income per capita:
- **File import and filtering**: The income and population tables are delivered in long format (one row per variable and period). Only the relevant variable is kept (gross disposable income of households and total population), and only annual figures are selected; quarterly figures are removed.
- **Type checking**: Years are extracted from the CBS period codes (e.g. `1999JJ00`) and values are converted to floats. For the overall CPI, decimal commas are replaced with points.
- **Merging**: The three series are joined by year, keeping only the years available in all sources (1999–2025).
- **Calculation**: Real disposable income per capita is calculated by dividing total disposable income by the population and deflating it with the overall CPI, expressing all values in 2015 euros:

$$\text{Real income per capita}_t = \frac{\text{Disposable income}_t}{\text{Population}_t \cdot \text{CPI}_t / 100}$$

- **Export**: The resulting dataset is saved as `income_data.csv` in `data/processed_data/` via `export_dataframe`.

Real disposable income per capita increased from about €19,300 in 1999 to €24,000 in 2025 (in 2015 prices), with a decline after the 2008 financial crisis and a temporary dip in 2022 due to high inflation.

# 4. Sub question 1: Evolution of the fuel-to-fare ratio

_How has the ratio between fuel prices and train ticket prices evolved over the last 25 years?_

## 4.1 Method

The evolution of the price ratio is based on the CPI series from CBS table 83131NED (see Section 3.1). The CPI of passenger transport by train represents the price of rail travel. The price of fuel is represented by a weighted average of the CPI of petrol, diesel and LPG. For 2021–2025 the 2020 weights are used, as more recent weights are not available. Both series use 2015 as base year, so the ratio is not affected by general inflation.

A value above 1 means that train fares have risen faster than fuel prices since 2015, and a value below 1 means that fuel has become relatively more expensive compared with train fares. The ratio compares price changes and excludes fixed car costs, discounted train tickets and changes in fuel efficiency.

## 4.2 Results

Figure X shows the monthly and annual ratio. Over 1999–2025 the annual ratio ranges from 0.76 (2022) to 1.09 (2016), with a mean of 0.96 and a standard deviation of 0.09. There is no trend over time, but there are several notable periods:

- **1999–2012:** a decline from about 1.10 to 0.79, so fuel became relatively more expensive.
- **2012–2020:** a recovery to about 1.10 in 2016, followed by fluctuation around 1.0 and a second peak of 1.10 in 2020.
- **2020–2022:** a sharp fall to 0.76, consistent with the fuel price peak in 2022.
- **2022–2025:** a partial recovery to about 0.96.

![Fare-to-fuel price ratio, 1999–2025](figures/2_fare_fuel_ratio.png)

*Figure X: Monthly and annual mean fare-to-fuel price ratio (train CPI divided by fuel CPI), 1999–2025. Source: CBS table 83131NED, own calculation.*

## 4.3 Answer to sub question 1

Between 1999 and 2025 the fare-to-fuel ratio fluctuated between 0.76 and 1.10 without a trend over the whole period. It fell until 2012, fluctuated around 1.0 until 2020, when the pandemic occurred, dropped sharply in 2022 because of the fuel price peak, and has been recovering slowly until 2025. However, the recent situation in Iran is not taken into account in this study.

# 5. Sub question 2: Effect of the price ratio on the modal split

_To what extent does a change in the fuel-to-fare price ratio affect the modal split between car and train?_

## 5.1 Method

To analyse the relationship between relative prices and mode choice, the constructed fare-to-fuel ratio from section 4 is combined with the modal split derived from table 84755NED, 1999–2024. The modal split is defined as the share of the train kilometres in the combined distance travelled by train and car.

The relationship is first inspected visually, in a time series and a scatter plot, and then quantified with a linear regression of the train share on the ratio, in the same year. The COVID years 2020 and 2021 are excluded from the regression, because lockdowns and working from home caused a drop in train travel that is unrelated to prices. Other influences such as income are not controlled for in this section. They are added in Section 6.

## 5.2 Results

<div align="center">
<img src="figures/3_modal_split.png" alt="Modal split of car and train" width="500">
</div>

*Figure X: Modal split between car and train, 1999–2025.*

The figure shows the train share over time compared to the total use of train and car kilometres. The train share is stable at about 12% between 1999 and 2013, jumps to about 14% in 2014, this split stays pretty much unchanged till 2019. It then falls to 10% in 2020 and 9.0% in 2021, recovers to 10.5% in 2022, and returns to about 13.5% in 2023, after which it remains around that level.

<div align="center">
<img src="figures/4_ratio_en_treinaandeel.png" alt="Price ratio and train share over time" width="500">
</div>

*Figure X: Fare-to-fuel ratio and train share of car and train kilometres over time. The grey band marks the COVID-19 period.*

The two series do not show a consistent common pattern. Between 1999 and 2012 the fare-to-fuel ratio changes considerably, while the train share stays close to 12%. The largest change in the train share, in 2014, takes place in a year in which the ratio changes only moderately. The year after that, in 2015, the fare-to-fuel price ratio goes up a lot with little change in mode use.

<div align="center">
<img src="figures/5_scatter_ratio_vs_share.png" alt="Train share versus fare-to-fuel ratio" width="500">
</div>

*Figure X: Train share of car and train kilometres against the fare-to-fuel ratio. The dashed line is the linear fit excluding the COVID years 2020 and 2021.*

The fitted line (excluding 2020 and 2021) in the scatterplot shows a slope of +5.3 percentage points per unit of the ratio. The relation that is found is opposite to the expected direction, a fare price growth in relation with fuel price indicates more train use.

## 5.3 Answer to sub question 2

A change in the fare-to-fuel ratio does not clearly affect the modal split between car and train. The association between the two is weakly positive, about +0.53 percentage points per 0.1 increase in the ratio, contrary to expectations. The train share is mainly shaped by other developments, such as the sharp drop during the COVID period. Whether this reflects the absence of a price effect or the influence of other factors is investigated in Section 6, using control variables.

# 6. Sub question 3: Price elasticity of modal choice

_What is the price elasticity of modal choice with respect to the fuel-fare ratio?_

## 6.1 Method

*To be completed.* 

*subsection on new controls*

### Control variables
A simple log-log regression of travel behaviour on the fare-to-fuel ratio may capture shared trends rather than a price effect: both car and train use can rise over time because of economic growth, independent of relative prices. To separate the price effect from these trends, the elasticity is re-estimated with a multiple log-log regression:

$$\ln(y_t) = \beta_0 + \beta_1 \ln(\text{Ratio}_t) + \beta_2 \ln(\text{Income}_t) + \beta_3 \, \text{COVID}_t + \varepsilon_t$$

where $y_t$ is train kilometres, car kilometres or the train share of car and train kilometres in year $t$, and $\beta_1$ is the elasticity with respect to the fare-to-fuel ratio. Two control variables are included:
- **Real disposable income per capita** (see Section 3.2.3), since income affects both the total amount of travel and the choice between car and train.
- **A COVID-19 dummy**, equal to 1 in pandemic years and 0 otherwise. Unlike the earlier approach, where COVID years were removed, the dummy keeps these years in the sample while preventing the sharp drop in travel from being attributed to prices.

Four models are compared, adding the controls step by step: (1) ratio only, (2) ratio and income, (3) ratio, income and a COVID dummy for 2020–2021, and (4) the same model with 2022 also included in the COVID period. Model 4 serves as a robustness check, because travel behaviour, and train use in particular, had not yet recovered to pre-pandemic levels in 2022. All models use the same 27 years (1999–2025). Given the small number of observations, the number of control variables is deliberately limited.

## 6.2 Results

*To be completed.*

*Subsection on new controls*

### Elasticity with control variables
Table X and Figure X show the estimated elasticities for the four models.

| Dependent variable | (1) Ratio only | (2) + income | (3) + income + COVID 2020–21 | (4) + income + COVID 2020–22 |
|---|---|---|---|---|
| Train km | 0.275 (± 0.74) | 0.147 (± 0.67) | 0.470 (± 0.31) | [..] (± [..]) |
| Car km | 0.143 (± 0.40) | 0.035 (± 0.28) | 0.160 (± 0.15) | [..] (± [..]) |
| Train share | 0.115 (± 0.41) | 0.097 (± 0.42) | 0.272 (± 0.28) | [..] (± [..]) |

*Values in parentheses are 95% confidence intervals.*

![Elasticity with and without control variables](code/figures/8_elasticiteit_met_controls.png)

*Bars show the estimated elasticity; whiskers show the 95% confidence interval. Intervals crossing zero indicate no statistically significant effect.*

Adding income mainly affects car use: the elasticity of car kilometres falls from 0.14 to 0.04, while the explained variance (R²) rises from 0.02 to 0.57. This confirms that the positive relation between the ratio and car use found earlier was largely driven by income growth rather than by prices.

Adding a COVID dummy for 2020–2021 strongly increases the explained variance (R² of 0.85 for train kilometres) and results in a significant positive elasticity for train use (0.47). This positive sign contradicts economic expectations. It can be explained by 2022: in that year the fare-to-fuel ratio reached its lowest value (0.76) due to the fuel price peak, while train use was still recovering from the pandemic. When 2022 is also treated as a COVID year (model 4), all elasticities drop to approximately zero and none is statistically significant.

## 6.3 Answer to sub question 3

*To be completed.*

*subconclusion on controls* 
After controlling for real income per capita and the COVID-19 period, the price elasticity of car and train use with respect to the fare-to-fuel ratio is close to zero and not statistically significant. The positive elasticities found in the simple regression were driven by shared trends in income and by the slow recovery of train use in 2022, not by relative prices. Within the range of ratios observed in the Netherlands between 1999 and 2025 (0.76–1.17), the relative price of train tickets versus fuel does not appear to be a dominant driver of mode choice at the national level.

These results should be interpreted with caution. With only 27 annual observations, small effects cannot be detected: an elasticity of around −0.2 still falls within the confidence intervals. In addition, consecutive years are strongly correlated, which may lead to underestimated standard errors. Non-monetary factors such as travel time, reliability and comfort, which were not included, may play a larger role in mode choice than the price ratio.

# 7. Conclusion

*To be completed. Summarise the answers to the three sub questions and use them to answer the main research question.*

# 8. Discussion

*To be completed. Reflect on the results, their limitations, and suggestions for further research.*

# References

Centraal Bureau voor de Statistiek. (2024). *Consumentenprijzen; prijsindex 2015=100* (Dataset 83131NED) [Data set]. StatLine. https://opendata.cbs.nl/#/CBS/nl/dataset/83131NED/table

Centraal Bureau voor de Statistiek. (2024). *Reizigersvervoer per spoor; reizigerskilometers* (Dataset 84755NED) [Data set]. StatLine. https://opendata.cbs.nl/#/CBS/nl/dataset/84755NED/table

Centraal Bureau voor de Statistiek. (2021). *Verkeersprestaties personenauto's; kilometers, grondgebied 1990-2020* (Dataset 80428NED) [data set]. statline. https://opendata.cbs.nl/#/CBS/nl/dataset/80428ned/table

Compendium voor de Leefomgeving. (2026). *Aantal motorvoertuigen actief, 2019-2026* (Indicator 0026, versie 28) [Data set]. CBS, PBL, RIVM, WUR. https://www.clo.nl/indicatoren/nl002628-aantal-motorvoertuigen-actief-2019-2026

# Contribution Statement

All authors contributed to the coding of this project. Below is a breakdown of each author's primary focus areas.

**Daan Devenijns:**
- Background research (literature on fuel price volatility and modal choice)
- Data collection (fuel prices)
- Coding

**Maarten Hogendoorn:**
- Data collection (train ticket prices / fare data)
- Data modelling (price elasticity estimation)
- Coding

**Hidde Okhuijsen:**
- Conceptualisation (research question and sub-questions)
- Data analysis
- Coding

**Gijs Six:**
- Data pipeline (cleaning and merging data sources)
- Visualisation
- Coding

**Nova Swanenburg:**
- Background research (modal split literature)
- Visualisation
- Coding

*Note: task division is preliminary and will be refined as the project progresses.*